# YOLOv8-Large Wafer Defect Detection — Part 2 (Steps 7–12)

Continues from the completed A100 training run in Part 1.

## Results Already Captured (Part 1 — 100 epochs on A100)

| Metric | Value |
|--------|-------|
| GPU | NVIDIA A100-SXM4-40GB |
| Model | YOLOv8-Large (43.7M params) |
| Epochs | 100 |
| **mAP@50** | **0.9922** |
| **mAP@50:95** | **0.9576** |
| **Precision** | **0.9907** |
| **Recall** | **0.9861** |
| Training time | 341.9 minutes |

> **Run Cell 2 FIRST** — it checks that `best.pt` still exists in `/content/`.
> If the Colab runtime was disconnected, weights are gone and you must re-run Part 1.
> **Run Cell 3 immediately after** — it backs up `best.pt` to Google Drive.

## Step 0A — Verify Runtime Is Still Alive

Fail fast if `best.pt` is missing.

In [ ]:
import subprocess, os, sys
from pathlib import Path

result = subprocess.run(
    ['find', '/content', '-name', 'best.pt', '-type', 'f'],
    capture_output=True, text=True
)
best_pts = [p for p in result.stdout.strip().split('\n') if p]

print('=== RUNTIME STATUS ===')
print(f'best.pt files found: {best_pts}')
print(f'Repo exists: {Path("/content/yolo-object-detection").exists()}')
print(f'Dataset yaml: {Path("/content/yolo-object-detection/data/wafer_defects/data.yaml").exists()}')

if not best_pts:
    raise RuntimeError(
        'Runtime was disconnected — weights lost.\n'
        'Re-run Part 1 notebook to re-train. Then come back here.'
    )

# Prefer the yolov8l_wafer run directory if multiple matches
BEST_PT = next((p for p in best_pts if 'yolov8l_wafer' in p), best_pts[0])
print(f'\nUsing: {BEST_PT}  ({Path(BEST_PT).stat().st_size / 1e6:.1f} MB)')
print('Runtime alive — proceed to backup!')


## Step 0B — Backup best.pt to Google Drive (do this FIRST)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/drive')

drive_dir = Path('/drive/MyDrive/yolo_wafer_defect_a100')
drive_dir.mkdir(parents=True, exist_ok=True)

dest = drive_dir / 'yolov8l_wafer_best.pt'
shutil.copy(BEST_PT, dest)
print(f'Model backed up: {dest}  ({dest.stat().st_size / 1e6:.1f} MB)')

# Backup any training plots that exist
run_dir = Path(BEST_PT).parent.parent
for f in list(run_dir.glob('*.png')) + list(run_dir.glob('*.csv')):
    shutil.copy(f, drive_dir / f.name)
    print(f'  Backed up: {f.name}')

print('\nBackup complete. Safe to continue with remaining steps.')


## Step 1 — Reinstall Packages + Restore Variables

The Colab kernel was reset after the 5.7-hour session. Re-install `ultralytics` and restore all constants from the completed A100 run.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'mlflow', 'onnx', 'onnxruntime-gpu'], check=True)

import os, sys, time, json, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import torch
import mlflow

os.chdir('/content/yolo-object-detection')
sys.path.insert(0, '.')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── constants ──────────────────────────────────────────────────────────
DATA_YAML  = 'data/wafer_defects/data.yaml'
BATCH_SIZE = 16
WORKERS    = 4

CLASSES = [
    'scratch', 'particle', 'edge_chip', 'void', 'pattern_shift',
    'bridge', 'missing_bond', 'crack', 'contamination', 'delamination',
]

# Hardcoded from completed Cell 13 output
L_RESULTS = {
    'mAP50': 0.9922, 'mAP50_95': 0.9576,
    'precision': 0.9907, 'recall': 0.9861,
    'training_time_min': 341.9,
}

PER_CLASS_RESULTS = {
    'scratch':       {'mAP50': 0.979,  'mAP50_95': 0.878},
    'particle':      {'mAP50': 0.995,  'mAP50_95': 0.938},
    'edge_chip':     {'mAP50': 0.995,  'mAP50_95': 0.984},
    'void':          {'mAP50': 0.995,  'mAP50_95': 0.995},
    'pattern_shift': {'mAP50': 0.995,  'mAP50_95': 0.995},
    'bridge':        {'mAP50': 0.984,  'mAP50_95': 0.845},
    'missing_bond':  {'mAP50': 0.995,  'mAP50_95': 0.994},
    'crack':         {'mAP50': 0.994,  'mAP50_95': 0.967},
    'contamination': {'mAP50': 0.995,  'mAP50_95': 0.987},
    'delamination':  {'mAP50': 0.995,  'mAP50_95': 0.993},
}

Path('outputs').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

print('\nVariables restored from completed A100 training:')
print(f'  mAP@50={L_RESULTS["mAP50"]:.4f}  mAP@50:95={L_RESULTS["mAP50_95"]:.4f}')
print(f'  Training time: {L_RESULTS["training_time_min"]:.1f} min  |  Model: {BEST_PT}')


## Step 7 — Model Size Comparison (S vs M vs L)

Train YOLOv8-S and YOLOv8-M for 20 epochs as baselines.
YOLOv8-L was already trained for 100 epochs — results are hardcoded above.

In [ ]:
mlflow.set_tracking_uri('file:///content/yolo-object-detection/mlruns')
mlflow.set_experiment('wafer-defect-comparison')

comparison_results = {}

for model_file, label, batch in [('yolov8s.pt', 'YOLOv8-S', 32),
                                  ('yolov8m.pt', 'YOLOv8-M', 16)]:
    print(f'\n{"="*50}')
    print(f'Training {label} — 20 epochs')
    print('='*50)
    comp_model = YOLO(model_file)
    if mlflow.active_run():
        mlflow.end_run()
    with mlflow.start_run(run_name=f'{label.lower().replace("-","")}-comparison'):
        t0 = time.time()
        r = comp_model.train(
            data=DATA_YAML, epochs=20, imgsz=640,
            batch=batch, patience=10, device=0,
            workers=WORKERS,
            project='runs/detect',
            name=f'{label.lower().replace("-","")}_wafer',
            exist_ok=True, mosaic=1.0, cos_lr=True,
        )
        elapsed = time.time() - t0
        comparison_results[label] = {
            'mAP50':            r.results_dict.get('metrics/mAP50(B)', 0),
            'mAP50_95':         r.results_dict.get('metrics/mAP50-95(B)', 0),
            'precision':        r.results_dict.get('metrics/precision(B)', 0),
            'recall':           r.results_dict.get('metrics/recall(B)', 0),
            'training_time_min': round(elapsed / 60, 1),
        }
        mlflow.log_metrics(comparison_results[label])

if mlflow.active_run():
    mlflow.end_run()

# Add L results (100 epochs)
comparison_results['YOLOv8-L'] = L_RESULTS.copy()

print(f'\n{"="*72}')
print(f'{"Model":<12} {"mAP@50":>10} {"mAP@50:95":>12} {"Precision":>10} {"Recall":>10} {"Time":>8}')
print('-'*72)
for name, res in comparison_results.items():
    ep = '(100ep)' if name == 'YOLOv8-L' else '(20ep) '
    print(f'{name:<12} {res["mAP50"]:>10.4f} {res["mAP50_95"]:>12.4f} '
          f'{res.get("precision",0):>10.4f} {res.get("recall",0):>10.4f} '
          f'{res["training_time_min"]:>6.1f}m {ep}')
print('='*72)


## Step 8 — Evaluate on Held-Out Test Set

3 000 unseen test images. Recall is critical for semiconductor inspection.

In [ ]:
best_model = YOLO(BEST_PT)

test_results = best_model.val(
    data=DATA_YAML, split='test',
    batch=16, device=0,
    plots=True, save_json=True,
)

print(f'\n{"="*50}')
print('TEST SET RESULTS  (3 000 unseen images)')
print('='*50)
rd = test_results.results_dict
print(f"  mAP@50:     {rd.get('metrics/mAP50(B)',    0):.4f}")
print(f"  mAP@50:95:  {rd.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision:  {rd.get('metrics/precision(B)',0):.4f}")
print(f"  Recall:     {rd.get('metrics/recall(B)',   0):.4f}")

if hasattr(test_results, 'maps'):
    print('\nPer-class mAP@50:')
    for i, cls_name in enumerate(CLASSES):
        if i < len(test_results.maps):
            print(f'  {cls_name:20s}: {test_results.maps[i]:.4f}')


## Step 9 — ONNX Export + Speed Benchmark

In [ ]:
# Export ONNX
print('Exporting to ONNX...')
best_model.export(format='onnx', dynamic=True, simplify=True, opset=17)

r2 = subprocess.run(
    ['find', '/content', '-name', 'best.onnx', '-type', 'f'],
    capture_output=True, text=True
)
BEST_ONNX = [p for p in r2.stdout.strip().split('\n') if p][0]

shutil.copy(BEST_PT,   'models/best.pt')
shutil.copy(BEST_ONNX, 'models/best.onnx')
print(f'PT  : {Path("models/best.pt").stat().st_size / 1e6:.1f} MB')
print(f'ONNX: {Path("models/best.onnx").stat().st_size / 1e6:.1f} MB')

# Save ONNX to Drive
drive_dir = Path('/drive/MyDrive/yolo_wafer_defect_a100')
if drive_dir.exists():
    shutil.copy('models/best.onnx', drive_dir / 'yolov8l_wafer_best.onnx')
    print('ONNX copied to Google Drive')

# Speed benchmark
dummy_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
N_WARMUP, N_RUNS = 10, 50
results_bench = {}

for fmt, path in [('PyTorch', 'models/best.pt'), ('ONNX', 'models/best.onnx')]:
    m = YOLO(path)
    for _ in range(N_WARMUP):
        m(dummy_img, verbose=False)
    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        m(dummy_img, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    results_bench[fmt] = {
        'mean_ms': round(float(np.mean(latencies)), 1),
        'p50_ms':  round(float(np.median(latencies)), 1),
        'p95_ms':  round(float(np.percentile(latencies, 95)), 1),
        'fps':     round(1000 / float(np.mean(latencies)), 1),
    }
    print(f"{fmt:12s}: {results_bench[fmt]['mean_ms']}ms avg | {results_bench[fmt]['fps']} FPS")

# TensorRT FP16 (may not be available on all Colab GPUs)
try:
    print('\nExporting TensorRT FP16...')
    best_model.export(format='engine', device=0, half=True)
    r3 = subprocess.run(
        ['find', '/content', '-name', 'best.engine', '-type', 'f'],
        capture_output=True, text=True
    )
    trt_path = [p for p in r3.stdout.strip().split('\n') if p][0]
    trt_model = YOLO(trt_path)
    for _ in range(N_WARMUP):
        trt_model(dummy_img, verbose=False)
    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        trt_model(dummy_img, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    results_bench['TensorRT-FP16'] = {
        'mean_ms': round(float(np.mean(latencies)), 1),
        'p50_ms':  round(float(np.median(latencies)), 1),
        'p95_ms':  round(float(np.percentile(latencies, 95)), 1),
        'fps':     round(1000 / float(np.mean(latencies)), 1),
    }
    print(f"TensorRT-FP16: {results_bench['TensorRT-FP16']['mean_ms']}ms | "
          f"{results_bench['TensorRT-FP16']['fps']} FPS")
except Exception as e:
    print(f'TensorRT skipped (not available): {e}')

with open('outputs/speed_benchmark.json', 'w') as f:
    json.dump(results_bench, f, indent=2)
print('\nSaved: outputs/speed_benchmark.json')


## Step 10 — Publication-Quality Visualizations (3 figures)

In [ ]:
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13})
C3 = ['#3b82f6', '#22c55e', '#ef4444']

# ── Figure 1: Model comparison ─────────────────────────────────────────
models = ['YOLOv8-S', 'YOLOv8-M', 'YOLOv8-L']
m50_vals  = [comparison_results.get(m, {}).get('mAP50', 0)             for m in models]
m95_vals  = [comparison_results.get(m, {}).get('mAP50_95', 0)          for m in models]
time_vals = [comparison_results.get(m, {}).get('training_time_min', 0) for m in models]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, vals, ylabel, title in [
    (axes[0], m50_vals,  'mAP@50',    'mAP@50 by Model Size'),
    (axes[1], m95_vals,  'mAP@50:95', 'mAP@50:95 by Model Size'),
    (axes[2], time_vals, 'Minutes',   'Training Time'),
]:
    bars = ax.bar(models, vals, color=C3)
    ax.set_title(title, fontweight='bold'); ax.set_ylabel(ylabel)
    for bar, v in zip(bars, vals):
        label = f'{v:.3f}' if v < 10 else f'{v:.0f} min'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            label, ha='center', fontsize=10, fontweight='bold'
        )
plt.suptitle('YOLOv8 Model Comparison — Wafer Defect Detection', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Speed benchmark ──────────────────────────────────────────
fmts = list(results_bench.keys())
fps_vals = [results_bench[f]['fps'] for f in fmts]
ms_vals  = [results_bench[f]['mean_ms'] for f in fmts]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(fmts, fps_vals, color=C3[:len(fmts)])
ax.set_title('Inference Speed Comparison (A100)', fontsize=14, fontweight='bold')
ax.set_ylabel('FPS (higher is better)')
for bar, fps, ms in zip(bars, fps_vals, ms_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{fps:.1f} FPS\n({ms:.1f}ms)',
        ha='center', fontsize=10, fontweight='bold'
    )
plt.tight_layout()
plt.savefig('outputs/speed_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Per-class performance ───────────────────────────────────
cls_names = list(PER_CLASS_RESULTS.keys())
m50_cls  = [PER_CLASS_RESULTS[c]['mAP50']    for c in cls_names]
m95_cls  = [PER_CLASS_RESULTS[c]['mAP50_95'] for c in cls_names]
y = np.arange(len(cls_names))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_50 = ['#22c55e' if v >= 0.99 else '#f97316' if v >= 0.97 else '#ef4444' for v in m50_cls]
axes[0].barh(y, m50_cls, color=colors_50)
axes[0].set_yticks(y); axes[0].set_yticklabels(cls_names)
axes[0].set_xlabel('mAP@50'); axes[0].set_title('Per-Class mAP@50', fontweight='bold')
axes[0].set_xlim(0.80, 1.02)
for i, v in enumerate(m50_cls):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

axes[1].barh(y, m95_cls, color='#3b82f6')
axes[1].set_yticks(y); axes[1].set_yticklabels(cls_names)
axes[1].set_xlabel('mAP@50:95'); axes[1].set_title('Per-Class mAP@50:95', fontweight='bold')
axes[1].set_xlim(0.70, 1.05)
for i, v in enumerate(m95_cls):
    axes[1].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('YOLOv8-L Per-Class Performance (A100, 100 epochs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print('3 figures saved to outputs/')


## Step 11 — Inference on Sample Test Images

In [ ]:
import matplotlib.patches as patches
from PIL import Image

PALETTE = [
    '#ef4444','#f97316','#eab308','#22c55e','#06b6d4',
    '#3b82f6','#8b5cf6','#ec4899','#f43f5e','#14b8a6',
]

test_imgs = sorted(Path('data/wafer_defects/test/images').glob('*.jpg'))[:8]
if not test_imgs:
    test_imgs = sorted(Path('data/wafer_defects/valid/images').glob('*.jpg'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, test_imgs):
    preds = best_model(str(img_path), conf=0.25, verbose=False)
    img = Image.open(img_path)
    ax.imshow(img)
    n_det = 0
    for r in preds:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            color  = PALETTE[cls_id % len(PALETTE)]
            ax.add_patch(
                patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
            )
            ax.text(
                x1, y1 - 4,
                f'{CLASSES[cls_id]} {conf:.2f}',
                fontsize=7, color='white', backgroundcolor=color
            )
            n_det += 1
    ax.set_title(f'{img_path.name[:20]}  ({n_det} det)', fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8-L Predictions on Test Images (conf >= 0.25)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/sample_predictions.png')


## Step 12 — Collect Artifacts + Download ZIP

In [ ]:
import zipfile

# Copy training plots from runs/ into outputs/
r4 = subprocess.run(
    ['find', '/content/yolo-object-detection/runs', '-name', '*.png', '-type', 'f'],
    capture_output=True, text=True
)
for pf in [p for p in r4.stdout.strip().split('\n') if p]:
    try:
        shutil.copy2(pf, f'outputs/{Path(pf).name}')
    except Exception:
        pass

# Build results summary
rd = test_results.results_dict
summary = {
    'model': 'YOLOv8-Large',
    'params_M': 43.7,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'A100-SXM4-40GB',
    'dataset': '20K synthetic wafer defects (10 classes)',
    'epochs': 100,
    'training_time_min': 341.9,
    'test_metrics': {
        'mAP50':     rd.get('metrics/mAP50(B)',     L_RESULTS['mAP50']),
        'mAP50_95':  rd.get('metrics/mAP50-95(B)',  L_RESULTS['mAP50_95']),
        'precision': rd.get('metrics/precision(B)', L_RESULTS['precision']),
        'recall':    rd.get('metrics/recall(B)',    L_RESULTS['recall']),
    },
    'per_class':        PER_CLASS_RESULTS,
    'model_comparison': comparison_results,
    'speed_benchmark':  results_bench,
}
with open('outputs/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=== FINAL METRICS ===')
print(f'  mAP@50:    {summary["test_metrics"]["mAP50"]:.4f}')
print(f'  mAP@50:95: {summary["test_metrics"]["mAP50_95"]:.4f}')
print(f'  Precision: {summary["test_metrics"]["precision"]:.4f}')
print(f'  Recall:    {summary["test_metrics"]["recall"]:.4f}')

print('\n=== OUTPUTS ===')
for ff in sorted(Path('outputs').glob('*')):
    print(f'  {ff.name:45s} {ff.stat().st_size / 1e3:7.0f} KB')

# ZIP
zip_path = '/content/yolov8_wafer_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ff in Path('outputs').glob('*'):
        zf.write(ff, f'outputs/{ff.name}')
    for ff in Path('models').glob('*'):
        zf.write(ff, f'models/{ff.name}')
print(f'\nZIP: {zip_path}  ({Path(zip_path).stat().st_size / 1e6:.1f} MB)')

# Save to Drive
drive_dir = Path('/drive/MyDrive/yolo_wafer_defect_a100')
if drive_dir.exists():
    shutil.copy(zip_path, drive_dir / 'yolov8_wafer_results.zip')
    shutil.copy('outputs/results_summary.json', drive_dir / 'results_summary.json')
    print('All artifacts saved to Google Drive')


In [ ]:
from google.colab import files
files.download('/content/yolov8_wafer_results.zip')
print('Download started. Check your browser downloads folder.')
